## Import libraries

In [2]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
from shapely.geometry import box
pd.set_option('display.max_columns', None)
place = "Berlin, Germany"

# Accident Data

## Load & Clean Accident Data

German column names are then renamed to descriptive English equivalents covering accident IDs, location codes, temporal attributes (year, month, hour, weekday), road/light conditions, vehicle involvement flags, and coordinates.

In [ ]:
df = pd.read_csv(
    "data/Raw/Unfallorte2024_LinRef.csv",
    sep=";",
    engine="python",
    dtype=str
)

df = df.replace("\t", "", regex=True)
df = df.apply(lambda col: col.str.strip())


rename_dict = {
    "OID_": "record_id",
    "UIDENTSTLAE": "accident_id",
    "ULAND": "state_code",
    "UREGBEZ": "administrative_region",
    "UKREIS": "district_code",
    "UGEMEINDE": "municipality_code",
    "UJAHR": "year",
    "UMONAT": "month",
    "USTUNDE": "hour",
    "UWOCHENTAG": "weekday",
    "UKATEGORIE": "accident_severity",
    "UART": "accident_type",
    "UTYP1": "detailed_accident_type",
    "ULICHTVERH": "light_condition",
    "IstStrassenzustand": "road_surface_condition",
    "IstRad": "involves_bicycle",
    "IstPKW": "involves_car",
    "IstFuss": "involves_pedestrian",
    "IstKrad": "involves_motorcycle",
    "IstGkfz": "involves_heavy_vehicle",
    "IstSonstige": "involves_other_vehicle",
    "LINREFX": "road_reference_x",
    "LINREFY": "road_reference_y",
    "XGCSWGS84": "longitude",
    "YGCSWGS84": "latitude",
    "PLST": "location_internal_code"
}

df = df.rename(columns=rename_dict)


In [ ]:
decimal_cols = [
    "road_reference_x",
    "road_reference_y",
    "longitude",
    "latitude"
]

for col in decimal_cols:
    df[col] = (
        df[col].str.replace(",", ".", regex=False)
        .astype(float)
    )

In [ ]:
int_cols = [
    "year", "month", "hour", "weekday",
    "accident_severity", "accident_type",
    "detailed_accident_type", "light_condition",
    "road_surface_condition",
    "involves_bicycle", "involves_car",
    "involves_pedestrian", "involves_motorcycle",
    "involves_heavy_vehicle", "involves_other_vehicle"
]

for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 5. Create datetime column
df["datetime"] = pd.to_datetime(
    df["year"].astype(str) + "-" +
    df["month"].astype(str) + "-01"
) + pd.to_timedelta(df["hour"], unit="h")

### Export Cleaned Accident Data

Saved the fully processed accident DataFrame

In [ ]:
df.to_csv("data//Processed/processed_accidents_2024.csv", index=False)

# OSM Street data

## Download and Enrich OSM Road Network

Fetched the drivable road graph for Berlin from OpenStreetMap using `osmnx`. Nodes and edges are extracted and reprojected to EPSG 25833 (UTM Zone 33N) so that distances are in metres.

Several features are derived for each edge:
- **`length_m`** - true planar length of the road segment in metres.
- **`road_type`** - first element of the highway tag (some edges carry multiple classifications).
- **`oneway`** - Boolean flag cast to integer.
- **`lanes`** - number of lanes, defaulting to 1 where missing.
- **`speed_limit`** - extracted from `maxspeed` with a fallback of 30 km/h.
- **`dist_to_intersection`** - minimum distance from each edge midpoint to the nearest intersection node.

In [6]:
G = ox.graph_from_place(place, network_type="drive")

nodes, edges = ox.graph_to_gdfs(G)
edges = edges.to_crs("EPSG:25833")
nodes = nodes.to_crs("EPSG:25833")

edges["length_m"] = edges.geometry.length

def clean_highway(x):
    if isinstance(x, list):
        return x[0]
    return x

edges["road_type"] = edges["highway"].apply(clean_highway)
edges["oneway"] = edges["oneway"].astype(int)
edges["lanes"] = pd.to_numeric(edges["lanes"], errors="coerce")
edges["lanes"] = edges["lanes"].fillna(1)
edges["maxspeed"] = edges["maxspeed"].astype(str)

edges["speed_limit"] = edges["maxspeed"].str.extract("(\d+)").astype(float)

edges["speed_limit"] = edges["speed_limit"].fillna(30)
G_u = G.to_undirected()

intersections = nodes[nodes["street_count"] >= 3]

edges["midpoint"] = edges.geometry.centroid

edges["dist_to_intersection"] = edges["midpoint"].apply(
    lambda x: intersections.distance(x).min()
)

<>:20: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:20: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Ranjith Panicker\AppData\Local\Temp\ipykernel_34720\676977411.py:20: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  edges["speed_limit"] = edges["maxspeed"].str.extract("(\d+)").astype(float)


# Combine data

## Integrate Berlin Bicycle Network

Loaded the official Berlin bicycle traffic network GeoJSON and reproject it to match the road edges CRS. Each bike-network segment is buffered by 10 m to account for small positional offsets between OSM and the official dataset.

Three complementary indicators are computed for every road edge:
- **`is_bike_network`** - 1 if the edge intersects the buffered bike-network union.
- **`bike_network_type`** - the specific network category via a spatial join.
- **`near_bike_lane`** - binary flag indicating whether the edge is within 20 m of any bike infrastructure.

In [7]:
bike_net = gpd.read_file("data/Processed/Bicycle traffic network-2025.json")
bike_net = bike_net.to_crs(edges.crs)

bike_net = bike_net.rename(columns={
    "ist_radvorrangnetz":"bike_network_type"
})

bike_buffer_distance = 10  # meters

bike_net["geometry_buffer"] = bike_net.geometry.buffer(bike_buffer_distance)

bike_buffer = gpd.GeoDataFrame(
    bike_net[["bike_network_type"]],
    geometry=bike_net["geometry_buffer"],
    crs=bike_net.crs
)


edges["is_bike_network"] = edges.geometry.intersects(
    bike_buffer.unary_union
).astype(int)

joined = gpd.sjoin(edges, bike_buffer, how="left", predicate="intersects")

network_type = joined.groupby(["u","v","key"])["bike_network_type"].first()

edges["bike_network_type"] = edges.index.map(network_type)

edges["bike_network_type"] = edges["bike_network_type"].fillna("none")

bike_union = bike_net.unary_union

edges["has_bike_infra"] = (edges.geometry.distance(bike_union) < 5).astype(int)
edges["dist_to_bike_network"] = edges.geometry.distance(bike_union)

# edges["dist_to_bike_network"] = edges["midpoint"].distance(bike_union)
edges["near_bike_lane"] = (edges["dist_to_bike_network"] < 20).astype(int)

C:\Users\Ranjith Panicker\AppData\Local\Temp\ipykernel_34720\3112950825.py:20: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  bike_buffer.unary_union
C:\Users\Ranjith Panicker\AppData\Local\Temp\ipykernel_34720\3112950825.py:31: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  bike_union = bike_net.unary_union


In [9]:
df = pd.read_csv("data/Processed/processed_accidents_2024.csv")

accidents = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326"
)

accidents = accidents.to_crs(edges.crs)
accidents = accidents[accidents["involves_bicycle"] == 1]
edges
joined = gpd.sjoin_nearest(
    accidents,
    edges,
    max_distance=30
)

crash_counts = joined.groupby(["u","v","key"]).size()

edges["crash_count"] = edges.index.map(crash_counts)

edges["crash_count"] = edges["crash_count"].fillna(0)
edges["crash"] = (edges["crash_count"] > 0).astype(int)
city = ox.geocode_to_gdf("Berlin, Germany").to_crs(edges.crs)


C:\Users\Ranjith Panicker\AppData\Local\Temp\ipykernel_34720\1170844244.py:1: DtypeWarning: Columns (0: accident_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/Processed/processed_accidents_2024.csv")


# Feature Creation

## Spatial Grid and Area-Level Feature Engineering

A 250 m x 250 m regular grid is built over the bounding box of Berlin, then clipped to the city boundary via `gpd.overlay`. Each grid cell receives a unique `grid_id`.

Road and bicycle network segments are overlaid with the grid to compute per-cell totals:
- **`road_length`** - total road length (metres) in the cell.
- **`bike_length`** - total bike-network length (metres) in the cell.
- **`intersection_count`** - number of OSM nodes with 3+ connections (proxy for road complexity).

In [10]:
xmin, ymin, xmax, ymax = city.total_bounds

cell_size = 250

cols = np.arange(xmin, xmax, cell_size)
rows = np.arange(ymin, ymax, cell_size)

polygons = []

for x in cols:
    for y in rows:
        polygons.append(box(x, y, x + cell_size, y + cell_size))

grid = gpd.GeoDataFrame({"geometry": polygons}, crs=city.crs)

grid = gpd.overlay(grid, city)

grid["grid_id"] = grid.index
road_grid = gpd.overlay(edges, grid)

road_grid["fragment_length"] = road_grid.geometry.length

road_length = road_grid.groupby("grid_id")["fragment_length"].sum()

grid["road_length"] = grid["grid_id"].map(road_length).fillna(0)
bike_grid = gpd.overlay(bike_net, grid)

bike_grid["fragment_length"] = bike_grid.geometry.length

bike_length = bike_grid.groupby("grid_id")["fragment_length"].sum()

grid["bike_length"] = grid["grid_id"].map(bike_length).fillna(0)
nodes["intersection"] = nodes["street_count"] >= 3

node_grid = gpd.sjoin(nodes, grid)

intersections = node_grid.groupby("index_right").size()

grid["intersection_count"] = grid.index.map(intersections).fillna(0)

## Assign Grid Features to Edges and Classify Road Types

Edge midpoints are spatially joined to the grid to inherit cell-level aggregate features (`grid_road_length`, `grid_bike_length`, `grid_intersections`). A helper function then maps OSM highway tags to five broad road classes used as model features: `major`, `secondary`, `tertiary`, `residential`, and `other`.

In [11]:
edges_original=edges.copy()

edges=edges.reset_index()
midpoints = gpd.GeoDataFrame(
    edges[["u","v","key"]],
    geometry=edges["midpoint"],
    crs=edges.crs
)

midpoints = gpd.sjoin(midpoints, grid)

edges = edges.merge(
    midpoints[["u","v","key","grid_id"]],
    on=["u","v","key"],
    how="left"
)
edges["grid_road_length"] = edges["grid_id"].map(
    grid.set_index("grid_id")["road_length"]
)

edges["grid_bike_length"] = edges["grid_id"].map(
    grid.set_index("grid_id")["bike_length"]
)

edges["grid_intersections"] = edges["grid_id"].map(
    grid.set_index("grid_id")["intersection_count"]
)
edges.shape, accidents.shape  
def road_class(x):

    if x in ["primary","primary_link"]:
        return "major"

    if x in ["secondary","secondary_link"]:
        return "secondary"

    if x in ["tertiary","tertiary_link"]:
        return "tertiary"

    if x in ["residential","living_street"]:
        return "residential"

    return "other"

edges["road_class"] = edges["road_type"].apply(road_class)



In [12]:
edges.drop(columns=["midpoint"], inplace=True)

In [50]:
edges.to_file("data/Processed/edges_with_exposure.geojson", driver="GeoJSON")

In [13]:
edges_=edges.copy()

# Compute Bicycle Traffic flow data

## Compute Average Daily Bicycle Traffic per Counter Station

Loaded the official Berlin automated bicycle counter data (hourly counts per station). The wide-format table is melted to long format, directional sub-stations are aggregated to station level, and daily totals are computed before taking the per-station annual mean (Average Daily Traffic, ADT). Station locations are merged from the metadata sheet and the result is converted to a GeoDataFrame.

In [14]:
traffic=pd.read_excel('data/Raw/gesamtdatei-stundenwerte.xlsx',sheet_name='Jahresdatei 2024')
traffic_location=pd.read_excel('data/Raw/gesamtdatei-stundenwerte.xlsx',sheet_name='Standortdaten')

traffic = traffic.rename(columns={traffic.columns[0]: "timestamp"})

traffic["timestamp"] = pd.to_datetime(traffic["timestamp"])

traffic.columns = [c.split("\n")[0] if "\n" in c else c for c in traffic.columns]


counts_long = traffic.melt(
    id_vars="timestamp",
    var_name="station_id",
    value_name="bike_count"
)

counts_long["station_base"] = counts_long["station_id"].str[:-2]

station_hourly = (
    counts_long
    .groupby(["station_base", "timestamp"])["bike_count"]
    .sum()
    .reset_index()
)

station_hourly["date"] = station_hourly["timestamp"].dt.date

daily_counts = (
    station_hourly
    .groupby(["station_base", "date"])["bike_count"]
    .sum()
    .reset_index()
)

station_adt = (
    daily_counts
    .groupby("station_base")["bike_count"]
    .mean()
    .reset_index()
)

station_adt = station_adt.rename(columns={"bike_count":"avg_daily_bikes"})

traffic_location["station_base"] = traffic_location["Zählstelle"].str[:-2]
traffic_location['station_base']=traffic_location['station_base'].replace({'17-SK-BRE':'17-SZ-BRE'})

station_adt=station_adt.merge(traffic_location[['station_base','Breitengrad','Längengrad']].rename(columns={'Breitengrad':'latitude','Längengrad':'longitude'}),on='station_base',how='left')

stations_gdf = gpd.GeoDataFrame(
    station_adt,
    geometry=gpd.points_from_xy(station_adt.longitude, station_adt.latitude),
    crs="EPSG:4326"
)

stations_gdf.drop_duplicates(subset="station_base", inplace=True)


edges_ = edges_.to_crs("EPSG:25833")
stations_gdf = stations_gdf.to_crs("EPSG:25833")

# edges_counts = gpd.sjoin_nearest(
#     edges,
#     stations_gdf,
#     how="left",
#     distance_col="dist_to_counter"
# )

# edges_counts["bike_volume_log"] = np.log1p(edges_counts["avg_daily_bikes"])

In [15]:
# edges_counts=edges_counts.to_crs("EPSG:4326")
# edges_counts.to_file("data/Processed/master_set2.geojson", driver="GeoJSON")


## Compute Edge Attractiveness Score

Computed a heuristic attractiveness score for each road edge that captures how appealing it is for cycling. The score combines five normalised indicators multiplicatively:
1. **Bike-network membership** - large bonus for official cycling streets.
2. **Local bike-network density** - more bike infrastructure in the cell means higher flow.
3. **Intersection density** - well-connected areas attract more cyclists.
4. **Speed limit** - high-speed roads are less attractive for cycling.
5. **Distance to bike network** - roads far from infrastructure see less cycling.

Edges outside the official bike network receive an additional 0.6x penalty, and the minimum score is clipped to 0.05 to avoid zero weights.

In [ ]:
# Edge centroids
edges_["centroid"] = edges_.geometry.centroid

edge_centroids = gpd.GeoDataFrame(
    edges_[["centroid"]],
    geometry="centroid",
    crs=edges_.crs
)


edges_["grid_bike_norm"] = edges_["grid_bike_length"] / edges_["grid_bike_length"].max()
edges_["intersections_norm"] = edges_["grid_intersections"] / edges_["grid_intersections"].max()
edges_["speed_norm"] = edges_["speed_limit"] / edges_["speed_limit"].max()

edges_["dist_bike_norm"] = edges_["dist_to_bike_network"] / edges_["dist_to_bike_network"].max()



edges_["attractiveness"] = (
    (1 + 4 * edges_["is_bike_network"]) *              # strong boost for bike streets
    (1 + 0.8 * edges_["grid_bike_norm"]) *             # dense bike networks attract flow
    (1 + 0.4 * edges_["intersections_norm"]) *         # connected grids attract cycling
    (1 - 0.4 * edges_["speed_norm"]) *                 # high speed roads discourage cycling
    (1 - 0.5 * edges_["dist_bike_norm"])               # far from bike network → less cycling
)

edges_.loc[edges_["is_bike_network"] == 0, "attractiveness"] *= 0.6


edges_["attractiveness"] = edges_["attractiveness"].clip(lower=0.05)

In [ ]:
beta = 1.5
max_distance = 15000  # meters

edges_["bike_flow"] = 0

for _, station in stations_gdf.iterrows():

    d = edge_centroids.distance(station.geometry)
    d = d.where(d <= max_distance)

    influence = station["avg_daily_bikes"] / (1 + (d / 1000) ** beta)

    flow = influence * edges_["attractiveness"]
    edges_["bike_flow"] += flow.fillna(0)

### Scale and Log-Transform Bike Flow

The raw gravity-model scores are linearly rescaled so the maximum edge flow equals 3000 cyclists/day (calibrated to observed counter data). A log-transformed version (`bike_flow_log`) is also stored because crash probability typically follows a logarithmic relationship with exposure volume.

In [18]:
total_station_counts = stations_gdf["avg_daily_bikes"].sum()

scale_factor = 3000 / edges_["bike_flow"].max()
edges_["bike_flow"] = edges_["bike_flow"] * scale_factor


edges_["bike_flow_log"] = np.log1p(edges_["bike_flow"])

In [27]:
edges_copy=edges_.copy()

### Compute Nearest Counter Station Distance



In [ ]:
def nearest_station_info(point, stations):
    dists = stations.distance(point)
    idx = dists.idxmin()
    nearest = stations.loc[idx]
    
    return pd.Series({
        "dist_to_station": dists.min(),
        "station_lat": nearest.geometry.y,
        "station_lon": nearest.geometry.x
    })

edges_copy[["dist_to_station", "station_lat", "station_lon"]] = edge_centroids.geometry.apply(
    lambda x: nearest_station_info(x, stations_gdf)
)

In [36]:
station_adt.to_csv('station_df.csv',index=False)

In [ ]:
edges_["dist_to_station"] = edge_centroids.geometry.apply(
    lambda x: stations_gdf.distance(x).min()
)

### Export Final Model-Ready Dataset

Save the fully enriched edge GeoDataFrame - including crash labels, OSM attributes, grid features, bicycle network flags, estimated bike flow, and station distances - as the canonical input file for model training.

In [23]:
edges_.to_file("data/Processed/model_prep_data.geojson", driver="GeoJSON")

In [69]:
edges_.describe()

,u,v,key,lanes,oneway,length,length_m,speed_limit,dist_to_intersection,is_bike_network,dist_to_bike_network,near_bike_lane,crash_count,crash,grid_id,grid_road_length,grid_bike_length,grid_intersections,grid_bike_norm,intersections_norm,speed_norm,dist_bike_norm,attractiveness,bike_flow,bike_flow_log,dist_to_station
count,7.379800e+04,7.379800e+04,73798.000000,73798.000000,73798.000000,73798.000000,73798.000000,73798.000000,73798.000000,73798.000000,73798.000000,73798.000000,73798.000000,73798.000000,73794.000000,73794.000000,73794.000000,73794.000000,73794.000000,73794.000000,73798.000000,7.379800e+04,73794.000000,73798.000000,73798.000000,73798.000000
mean,1.034773e+09,1.032814e+09,0.007358,1.284195,0.164693,143.648486,143.897471,35.121304,64.501097,0.654652,79.085734,0.416637,0.073688,0.059433,7430.505841,1426.596749,270.966724,4.736876,0.229860,0.205951,0.292678,6.184245e-02,3.993631,327.101486,4.875063,3684.911169
std,2.345647e+09,2.341534e+09,0.085621,0.597575,0.370906,152.111929,152.367853,9.426753,56.001329,0.475485,97.060376,0.493005,0.332226,0.236434,3536.925433,460.899860,193.698594,3.111178,0.164314,0.135269,0.078556,7.589803e-02,2.541904,404.001420,1.582873,2664.189389
min,1.725390e+05,1.725390e+05,0.000000,1.000000,0.000000,0.718684,0.719191,3.000000,0.359595,0.000000,0.000163,0.000000,0.000000,0.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.025000,1.278420e-07,0.270000,0.000000,0.000000,0.428603
25%,2.971153e+07,2.971085e+07,0.000000,1.000000,0.000000,61.906475,62.020825,30.000000,29.823630,0.000000,1.199614,0.000000,0.000000,0.000000,4776.000000,1130.396360,111.212527,3.000000,0.094341,0.130435,0.250000,9.380587e-04,0.647784,38.411980,3.674070,1598.167646
50%,9.821984e+07,9.822585e+07,0.000000,1.000000,0.000000,114.763799,114.964662,30.000000,55.787620,1.000000,52.917366,0.000000,0.000000,0.000000,7273.000000,1452.347985,264.688843,4.000000,0.224535,0.173913,0.250000,4.137964e-02,5.286404,169.791378,5.140443,3045.408360
75%,6.611008e+08,6.610485e+08,0.000000,1.000000,0.000000,185.012428,185.302984,50.000000,86.969344,1.000000,123.448591,1.000000,0.000000,0.000000,10431.000000,1730.837589,398.557758,6.000000,0.338095,0.260870,0.416667,9.653274e-02,5.943646,449.619764,6.110624,5155.482563
max,1.361932e+10,1.361932e+10,2.000000,7.000000,1.000000,8035.082975,8044.163847,120.000000,1802.943100,1.000000,1278.826088,1.000000,12.000000,1.000000,14755.000000,2984.428061,1178.833184,23.000000,1.000000,1.000000,1.000000,1.000000e+00,9.507758,3000.000000,8.006701,15599.652488
